# Анализ данных 

In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/hakaton.csv", sep=";")
print(f"Строк: {len(df)}, столбцов: {len(df.columns)}")
df.head(3)

Строк: 25628, столбцов: 21


,last_name,first_name,middle_name,bdate,gender,id_doc,guard_last_name,guard_first_name,guard_middle_name,guard_bdate,...,guard_id_doc,our_number,ogrn_naprav,name_naprav,ogrn_area,name_area,variant,class,test_date,result
0,МУСЮС,АЛЕКСАНДР,NaN,2013-07-16,М,240045105,АГАФОНОВА,КЛАВДИЯ,NaN,1987-07-06,...,106077198701603,26-11-60078-000038-4,1022402487304,муниципальное автономное общеобразовательное у...,1142452001130,краевое бюджетное общеобразовательное учрежден...,20611,6,2026-02-25,Недостаточный
1,СОБОЛЕВ,ЮРИЙ,ГЕННАДЬЕВИЧ,2018-01-26,М,-1236233,КАБЛИН,ВАЛЕНТИН,ГЕННАДИЕВИЧ,1991-03-15,...,1571302,25-11-70122-000036-6,1025007390077,МУНИЦИПАЛЬНОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ УЧ...,1035004258970,МУНИЦИПАЛЬНОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ УЧ...,120126,1,2025-12-16,Недостаточный
2,РЯБОВ,ГЕННАДИЙ,СЕРГЕЕВИЧ,2014-08-21,М,5974549,ЗАКИРОВ,БОРИС,СЕРГЕЕВИЧ,1990-03-07,...,5480835,26-11-70285-000004-4,1037816029393,ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ ...,1037816029547,Государственное бюджетное общеобразовательное ...,10410,4,2026-01-20,Достаточный


In [21]:
import bvista

bvista.show(df)

## Типы данных и пропуски

In [22]:
# Приведение типов
df["bdate"] = pd.to_datetime(df["bdate"], errors="coerce")
df["guard_bdate"] = pd.to_datetime(df["guard_bdate"], errors="coerce")
df["test_date"] = pd.to_datetime(df["test_date"], errors="coerce")
df["class"] = pd.to_numeric(df["class"], errors="coerce").astype("Int64")

# Нормализация result: trim + lower
df["result"] = df["result"].str.strip().str.lower()
valid_results = {"достаточный", "недостаточный"}
df.loc[~df["result"].isin(valid_results), "result"] = pd.NA
df["result"] = df["result"].astype("category")

print("Типы столбцов после преобразования:")
print(df.dtypes)

Типы столбцов после преобразования:
last_name                       str
first_name                      str
middle_name                     str
bdate                datetime64[us]
gender                          str
id_doc                          str
guard_last_name                 str
guard_first_name                str
guard_middle_name               str
guard_bdate          datetime64[us]
guard_gender                    str
guard_id_doc                    str
our_number                      str
ogrn_naprav                   int64
name_naprav                     str
ogrn_area                     int64
name_area                       str
variant                         str
class                         Int64
test_date            datetime64[us]
result                     category
dtype: object


In [23]:
# Пропуски по каждому столбцу
null_counts = df.isnull().sum()
print(null_counts)
null_pct = (null_counts / len(df) * 100).round(2)

null_report = pd.DataFrame({
    "nulls": null_counts,
    "pct": null_pct
}).query("nulls > 0").sort_values("nulls", ascending=False)

print(f"Столбцы с пропусками ({len(null_report)} из {len(df.columns)}):")
print(null_report.to_string())

last_name               0
first_name              0
middle_name          4469
bdate                   0
gender                  0
id_doc                 33
guard_last_name         0
guard_first_name        0
guard_middle_name    7389
guard_bdate             0
guard_gender            0
guard_id_doc           76
our_number              0
ogrn_naprav             0
name_naprav             0
ogrn_area               0
name_area               0
variant                 0
class                   1
test_date               0
result                  0
dtype: int64
Столбцы с пропусками (5 из 21):
                   nulls    pct
guard_middle_name   7389  28.83
middle_name         4469  17.44
guard_id_doc          76   0.30
id_doc                33   0.13
class                  1   0.00


In [24]:
expected_nulls = {"middle_name", "guard_middle_name"}
unexpected = null_report[~null_report.index.isin(expected_nulls)]

if len(unexpected) == 0:
    print("Неожиданных пропусков нет")
else:
    print("Неожиданные пропуски:")
    print(unexpected.to_string())

Неожиданные пропуски:
              nulls   pct
guard_id_doc     76  0.30
id_doc           33  0.13
class             1  0.00


## Аномалии в `id_doc/guard_id_doc`

In [25]:
import re

def is_suspicious_id(val):
    """Возвращает True если значение не похоже на валидный номер документа"""
    if pd.isna(val):
        return False
    s = str(val).strip()
    if re.fullmatch(r"0+", s):          # 000000 - заглушка
        return True
    if s.startswith("-"):               # отрицательное число
        return True
    if re.search(r"[A-Za-z]", s):       # буквенно-числовое (N18005999)
        return True
    return False

df["id_doc_suspicious"] = df["id_doc"].apply(is_suspicious_id)
df["guard_id_doc_suspicious"] = df["guard_id_doc"].apply(is_suspicious_id)

child_susp = df["id_doc_suspicious"].sum()
guard_susp = df["guard_id_doc_suspicious"].sum()
print(f"Подозрительных id_doc: {child_susp} ({child_susp/len(df)*100:.2f}%)")
print(f"Подозрительных guard_id_doc: {guard_susp} ({guard_susp/len(df)*100:.2f}%)")

Подозрительных id_doc: 5211 (20.33%)
Подозрительных guard_id_doc: 1521 (5.93%)


In [26]:
# Примеры подозрительных значений
print("Примеры подозрительных id_doc")
print(df[df["id_doc_suspicious"]]["id_doc"].value_counts().head(10).to_string())

print("\nПримеры подозрительных guard_id_doc")
print(df[df["guard_id_doc_suspicious"]]["guard_id_doc"].value_counts().head(10).to_string())

Примеры подозрительных id_doc
id_doc
-             13
-767625        6
N15667234      5
N16691028      5
-№723526       4
-815448        4
N18207316      3
N13348374      3
-N№0747234     3
N17448578      3

Примеры подозрительных guard_id_doc
guard_id_doc
N16454946     7
N13197044     6
N1793635      6
N13069762     6
N349645       6
19N           5
№N14501073    5
N14756850     5
N4631168      5
N12491147     5


## Распределения ключевых полей

In [27]:
# class: ожидаемый диапазон 1–11
print("Распределение по классам")
print(df["class"].value_counts().sort_index().to_string())
out_of_range = df[(df["class"] < 1) | (df["class"] > 11)]
print(f"\nВне диапазона 1–11: {len(out_of_range)} записей")

Распределение по классам
class
1     12306
2      1839
3      1803
4      1762
5      1952
6      1570
7      1534
8      1354
9       972
10      388
11      147

Вне диапазона 1–11: 0 записей


In [28]:
# Возраст ребёнка и родителя на дату теста
today = pd.Timestamp("2026-04-17")
df["child_age"] = ((df["test_date"] - df["bdate"]).dt.days / 365.25).round(1)
df["guard_age"] = ((df["test_date"] - df["guard_bdate"]).dt.days / 365.25).round(1)

print("Возраст ребёнка")
print(df["child_age"].describe().round(1))

print("\nВозраст родителя")
print(df["guard_age"].describe().round(1))

# Подозрительные возрасты
too_young = df[df["child_age"] < 6]
too_old   = df[df["child_age"] > 18]
guard_young = df[df["guard_age"] < 18]

print(f"\nДети младше 6 лет:      {len(too_young)}")
print(f"Дети старше 18 лет:     {len(too_old)}")
print(f"Родители младше 18 лет: {len(guard_young)}")

Возраст ребёнка
count    25628.0
mean         9.7
std          2.9
min         -0.0
25%          7.3
50%          8.5
75%         11.8
max         33.6
Name: child_age, dtype: float64

Возраст родителя
count    25628.0
mean        37.8
std          6.2
min         19.6
25%         33.4
50%         37.1
75%         41.3
max         92.8
Name: guard_age, dtype: float64

Дети младше 6 лет:      15
Дети старше 18 лет:     16
Родители младше 18 лет: 0


In [29]:
# Родитель должен быть старше ребёнка минимум на 14 лет (как предложение)
df["parent_child_age_diff"] = df["guard_age"] - df["child_age"]
parent_too_young = df[df["parent_child_age_diff"] < 14]
print(f"Родитель старше ребёнка менее чем на 14 лет: {len(parent_too_young)}")
if len(parent_too_young) > 0:
    print(parent_too_young[["last_name", "first_name", "bdate", "guard_last_name", "guard_bdate", "parent_child_age_diff"]].head(5).to_string())

Родитель старше ребёнка менее чем на 14 лет: 26
       last_name first_name      bdate guard_last_name guard_bdate  parent_child_age_diff
752   САМОХОДКИН   ВЛАДИМИР 2011-10-03        НАЗАРОВА  2001-01-01                   10.8
1306   ПОЛИКАНОВ  АЛЕКСАНДР 2010-11-06       ПОЛИКАНОВ  1999-07-06                   11.3
2209     ГУБАНОВ     СЕРГЕЙ 2007-11-11          ЖУРАЕВ  1994-10-04                   13.1
6468   УГОЛЬКОВА      ИРИНА 2012-01-20        СТАРОЖУК  1998-05-03                   13.7
7024    МАТВЕЕВА  ВАЛЕНТИНА 2011-09-08        ГОРБАЧЕВ  1998-02-05                   13.6


In [30]:
# test_date: диапазон и выбросы
print("Диапазон дат тестирования")
print(f"Мин: {df["test_date"].min().date()}")
print(f"Макс: {df["test_date"].max().date()}")

future_tests = df[df["test_date"] > today]
print(f"\nДаты в будущем (после 2026-04-17): {len(future_tests)}")

# variant: распределение
print("\nТоп-10 вариантов по частоте")
print(df["variant"].value_counts().head(10).to_string())

Диапазон дат тестирования
Мин: 2025-04-03
Макс: 2026-03-17

Даты в будущем (после 2026-04-17): 0

Топ-10 вариантов по частоте
variant
80109     1742
80110     1485
90113     1234
90114      785
100117     743
100118     560
60105      503
110121     479
120125     465
60107      443
